# Test de l'environnement PySpark + Delta Lake
Ce notebook vérifie PySpark, Spark SQL et l'écriture/lecture d'une table Delta.


In [ ]:
import sys, pyspark, delta
print('Python:', sys.version)
print('PySpark:', pyspark.__version__)
print('Delta:', delta.__version__)


In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (SparkSession.builder
    .appName('PySpark-Learning-Lab')
    .master('local[*]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.warehouse.dir', '/workspace/warehouse')
    .config('spark.driver.memory', '2g'))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('Master:', spark.sparkContext.master)


In [ ]:
data = [(1, 'Alice', 25), (2, 'Youssef', 30), (3, 'Sara', 27)]
columns = ['id', 'nom', 'age']
df = spark.createDataFrame(data, columns)
df.show()
df.printSchema()


In [ ]:
df.createOrReplaceTempView('personnes')
spark.sql('SELECT nom, age FROM personnes WHERE age >= 27').show()


In [ ]:
delta_path = '/workspace/warehouse/demo_delta'
df.write.format('delta').mode('overwrite').save(delta_path)
delta_df = spark.read.format('delta').load(delta_path)
delta_df.show()


In [ ]:
from delta.tables import DeltaTable
table = DeltaTable.forPath(spark, delta_path)
table.update(condition='id = 1', set={'age': 'age + 1'})
spark.read.format('delta').load(delta_path).show()


In [ ]:
spark.stop()
print('Test terminé.')
